### 27.Style Transfer (Fast Neural Transfer)

In [ ]:
# Apply fast neural style transfer using a pretrained TensorFlow Hub model to blend the content of one image with the artistic style of another.

import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Load and preprocess images
def load_image(path, target_size=(256, 256)):
    img = Image.open(tf.keras.utils.get_file(path.split('/')[-1], path)).convert('RGB') # Load and convert
    img = img.resize(target_size) #resize to target size
    img = np.array(img) / 255.0 # Normalize to [0, 1]
    return tf.constant(img, dtype=tf.float32)

# Load content and style images
content_path = 'https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg'

style_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/Vassily_Kandinsky%2C_1913_-_Composition_7.jpg"

# Load content and style images
content_image = load_image(content_path)
style_image = load_image(style_url)

# Add batch dimensions
content_image = tf.expand_dims(content_image, axis=0)
style_image = tf.expand_dims(style_image, axis=0)

# Load the style transfer model from Tensorflow Hub
model = hub.load("https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2")

# Apply style transfer
stylized_image = model(content_image, style_image)[0] # Output image

# Display results
def show_image(img_tensor, title):
    img = tf.squeeze(img_tensor).numpy() # remove batch dimension
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
show_image(content_image, 'Content Image')
plt.subplot(1, 3, 2)
show_image(style_image, 'Style Image')

plt.subplot(1, 3, 3)
show_image(stylized_image, "Stylized")
plt.tight_layout()
plt.show()


### 28. Super-Resolution with EDSR

In [ ]:
# Use a pretrained EDSR (Enhanced Deep Super-Resolution) model from TensorFlow Hub to upscale a low-resolution image to high-resolution.

import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Load and preprocess images
def load_and_resize_image(url, scale=4):
    path = tf.keras.utils.get_file(url.split("/")[-1], url) # Download image
    img = Image.open(path).convert("RGB") # Open image and convert to RGB
    hr_img = img.resize((128, 128)) # resize image to high resolution
    lr_img = hr_img.resize((128 // scale, 128 // scale)) # resize image to low resolution
    lr_img = lr_img.resize((128, 128)) # resize low resolution image to 128x128
    return np.array(lr_img) / 255.0, np.array(hr_img) / 255.0 # Normalize to [0, 1] and return both images


# Load sample image
img_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"

lr_img, hr_img = load_and_resize_image(img_url)

# Convert low_res image to tensor and add batch dimension
lr_tensor = tf.convert_to_tensor(lr_img, dtype=tf.float32)[tf.newaxis, ...]

# Load EDSR model from TensorFlow Hub
model = hub.load("https://tfhub.dev/captain-pool/esrgan-tf2/1") # ESRGAN model is similar to EDSR

# Run Model to get super-res output
sr_tensor = model(lr_tensor) # Super resolution output
sr_img = tf.clip_by_value(sr_tensor[0], 0.0, 1.0).numpy() # Convert to numpy and clip values, remove batch dimension and clip value is to ensure the pixel values are between 0 and 1

# Plot results
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(lr_img)
plt.title("Low-Res Input")
plt.axis('off')
 
plt.subplot(1, 3, 2)
plt.imshow(hr_img)
plt.title("Original High-Res")
plt.axis('off')
 
plt.subplot(1, 3, 3)
plt.imshow(sr_img)
plt.title("Super-Res Output")
plt.axis('off')
plt.tight_layout()
plt.show()



### 29. Image Denoising Autoencoder

In [ ]:
# Build a convolutional autoencoder using TensorFlow 2 to remove noise from images. We'll use MNIST digits with added Gaussian noise as the dataset.

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D

# Load MNIST dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train = np.expand_dims(x_train, -1) # Add channel dimension
x_test = np.expand_dims(x_test, -1) # Add channel dimension

# Add Gaussian noise to training data
noise_factor = 0.5
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

x_train_noisy = np.clip(x_train_noisy, 0., 1.) # clip means pixel values are between 0 and 1
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

# Build convolutional autoencoder
input_img = tf.keras.Input(shape=(28, 28, 1))
x = Conv2D(32, 3, activation='relu', padding='same')(input_img)
x = MaxPooling2D(2, padding='same')(x)
x = Conv2D(16, 3, activation='relu', padding='same')(x)
x = MaxPooling2D(2, padding='same')(x)
x = Conv2D(16, 3, activation='relu', padding='same')(x)
x = UpSampling2D(2)(x)
x = Conv2D(32, 3, activation='relu', padding='same')(x)
x = UpSampling2D(2)(x)
decoded = Conv2D(1, 3, activation='sigmoid', padding='same')(x)

autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train the autoencoder
autoencoder.fit(x_train_noisy, x_train, epochs=5, batch_size=128, shuffle=True, validation_split=0.1)

# Predict denoised images
decoded_imgs = autoencoder.predict(x_test_noisy[:10])

# Display original vs denoised images
plt.figure(figsize=(20, 4))
for i in range(10):
    # Noisy
    ax = plt.subplot(2, 10, i + 1)
    plt.imshow(x_test_noisy[i].squeeze(), cmap='gray')
    plt.title("Noisy")
    plt.axis("off")
    
    # Denoised
    ax = plt.subplot(2, 10, i+11)
    plt.imshow(decoded_imgs[i].squeeze(), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")
plt.show()

In [ ]:
# Predict denoised images (Use 20 images instead of 10)
decoded_imgs = autoencoder.predict(x_test_noisy[:20])

# Display original vs denoised images
plt.figure(figsize=(20, 4))
for i in range(20):
    # Noisy images (Top row: 1 to 20)
    ax = plt.subplot(2, 20, i + 1)
    plt.imshow(x_test_noisy[i].squeeze(), cmap='gray')
    plt.title("Noisy")
    plt.axis("off")
    
    # Denoised images (Bottom row: 21 to 40)
    ax = plt.subplot(2, 20, i + 21)
    plt.imshow(decoded_imgs[i].squeeze(), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")

plt.tight_layout()
plt.show()


### 30. Depth Estimation from Monocular Images

In [ ]:
# Use a pretrained monocular depth estimation model from TensorFlow Hub to predict a depth map from a single RGB image.

import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

# --- 1. MODEL INITIALIZATION ---
# Load the PyTorch-converted MiDaS v2.1 small model.
# This legacy SavedModel structure requires explicit 'serve' tags.
MODEL_URL = "https://tfhub.dev/intel/midas/v2_1_small/1"
depth_model = hub.load(MODEL_URL, tags=['serve'])
infer_fn = depth_model.signatures['serving_default']


# --- 2. PREPROCESSING PIPELINE ---
def load_and_preprocess(image_url):
    """
    Downloads an image, scales it, and formats it to the NCHW shape format 
    expected by the MiDaS graph signature.
    """
    # Use the base filename from URL to prevent collision in local cache
    local_filename = os.path.basename(image_url)
    image_path = tf.keras.utils.get_file(local_filename, origin=image_url)
    
    # Open, standardize dimensions to 256x256, and convert to numpy float matrix
    pil_img = Image.open(image_path).convert('RGB').resize((256, 256)) 
    np_img = np.array(pil_img).astype(np.float32) / 255.0 
    
    # Convert to Tensor and attach Batch Dimension -> (1, 256, 256, 3) [NHWC]
    tensor_img = tf.convert_to_tensor(np_img)
    tensor_img = tf.expand_dims(tensor_img, 0) 
    
    # Transpose layout to Channels-First -> (1, 3, 256, 256) [NCHW]
    model_input = tf.transpose(tensor_img, perm=[0, 3, 1, 2])
    
    # Return both the model tensor and the native numpy image for quick plotting
    return model_input, np_img


# --- 3. EXECUTION ---
IMAGE_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"
input_tensor, visual_img = load_and_preprocess(IMAGE_URL)

# Run inference through the static graph signature layer
output_dict = infer_fn(input_tensor)

# Extract depth frame and drop unused batch wrappers -> shape (256, 256)
depth_map = output_dict['default'][0] 


# --- 4. POST-PROCESSING & NORMALIZATION ---
# Min-Max Normalization to clamp values into a visible [0.0, 1.0] color distribution
depth_min = tf.reduce_min(depth_map)
depth_max = tf.reduce_max(depth_map)
normalized_depth = (depth_map - depth_min) / (depth_max - depth_min)


# --- 5. VISUALIZATION ---
plt.figure(figsize=(10, 4))

# Subplot 1: Render the original image asset
plt.subplot(1, 2, 1)
plt.imshow(visual_img) # Directly plotting the saved array (no inverse transpose needed)
plt.title("Original Image")
plt.axis('off')

# Subplot 2: Render the normalized heatmap output
plt.subplot(1, 2, 2)
plt.imshow(normalized_depth.numpy(), cmap='inferno')
plt.title("Estimated Depth Map")
plt.axis('off')

plt.tight_layout()
plt.show()

### 31. Text Classification with IMDB

In [ ]:
# Train a binary text classification model using TensorFlow 2 to classify IMDB movie reviews as positive or negative.

import tensorflow as tf
import matplotlib.pyplot as plt

# Load IMDB dataset (pre-tokenized, 10k word vocab)
vocab_size = 10000
(X_train, y_train),(X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

# Pad sequences to the same length
maxlen = 200
X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train, maxlen=maxlen)
X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test, maxlen=maxlen)

# Build the model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=maxlen), # Word embedding
    
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)), # Bidirectional LSTM
    tf.keras.layers.Dense(64, activation='relu'), 
    tf.keras.layers.Dense(1, activation='sigmoid') # output: 0 (neg) or 1 (pos)
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=3, batch_size=64, validation_split=0.2)

# Evaluate on test data
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.2f}")

# Plot training / validation accuracy
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title("IMDB sentiment Classification")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION & DATASETS ---
vocab_size = 10000
maxlen = 200

# Load IMDB dataset (pre-tokenized)
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

# Standardize text length via padding/truncating
# Explicitly specifying 'post' padding/truncating is often preferred for LSTMs
X_train = tf.keras.preprocessing.sequence.pad_sequences(X_train, maxlen=maxlen, padding='post', truncating='post')
X_test = tf.keras.preprocessing.sequence.pad_sequences(X_test, maxlen=maxlen, padding='post', truncating='post')

# --- 2. ADVANCED MODEL ARCHITECTURE ---
model = tf.keras.Sequential([
    # Modern Keras standard: Define explicit input dimensions here
    tf.keras.Input(shape=(maxlen,)),
    
    # Embedding layer maps sparse token IDs to dense vectors
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=64),
    
    # Bi-LSTM returning sequences to allow full sequence evaluation
    # Added regular dropout and recurrent dropout to mitigate overfitting
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)
    ),
    
    # Extracts the most powerful temporal features across the entire timeline
    tf.keras.layers.GlobalMaxPooling1D(),
    
    # Fully connected decision head
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3), # Extra regularization before final classification
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Display updated topology
model.summary()

# --- 3. COMPILATION & TRAINING ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

# Train the model 
history = model.fit(
    X_train, y_train, 
    epochs=4, 
    batch_size=64, 
    validation_split=0.2
)

# --- 4. EVALUATION & VISUALIZATION ---
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n﷿﷿﷿﷿ Optimized Test Accuracy: {acc:.2f}")

# Plot results
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy', linestyle='--')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title("IMDB Sentiment Classification (Optimized Model)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 32. Tokenization with SubwordTextEncoder

In [ ]:
# Use TensorFlow Datasets﷿﷿﷿ SubwordTextEncoder to build a subword-level tokenizer, then encode and decode sentences for NLP tasks like translation or classification.
import tensorflow_datasets as tfds
import tensorflow as tf

# Sample Sentences
corpus = [
    "TensorFlow is an end-to-end open-source platform for machine learning.",
    "Natural Language Processing is a fascinating field.",
    "Tokenization is the first step in NLP pipelines.",
    "Subword tokenization helps with rare words."
]

# Build SubWordTextEncoder from corpus
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    corpus, target_vocab_size=1000
)

# Print vocabulary size
print("SubWord Vocabulary Size: ", tokenizer.vocab_size)

# Encoder and decode a test sentence
test_sentence = "Subword tokenization is powerful for text models."
encoded = tokenizer.encode(test_sentence)
decoded = tokenizer.decode(encoded)

# Display results
print("\nOriginal Sentence:\n", test_sentence)
print("\nEncoded Tokens:\n", encoded)
print("\nDecoded Sentence:\n", decoded)
 
# Optional: visualize subwords
print("\nSubword Tokens:")
print([tokenizer.decode([token]) for token in encoded])

###  33. Bidirectional LSTM for Sentiment Analysis

In [ ]:
# Build a bidirectional LSTM model using TensorFlow 2 to perform sentiment classification on IMDB movie reviews (positive or negative).

import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import datasets, preprocessing, layers, Sequential

# Load IMDB dataset with top 10,000 most frequent words
vocab_size = 10000
maxlen = 300
(X_train, y_train), (X_test, y_test) = datasets.imdb.load_data(num_words=vocab_size)

# Pad Sequences to a uniform length
X_train = preprocessing.sequence.pad_sequences(X_train, maxlen=maxlen)
X_test = preprocessing.sequence.pad_sequences(X_test, maxlen=maxlen)

# Define Bi-LSTM Model
model = Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=128, input_length=maxlen),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
 
# Train the model
history = model.fit(X_train, y_train, epochs=3, batch_size=64, validation_split=0.2)
 
# Evaluate on test set
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.2f}")
 
# Plot training history
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title("Bidirectional LSTM - IMDB Sentiment")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


### 34.Word Embeddings with Word2Vec (Custom Training)

In [ ]:
import tensorflow as tf
import numpy as np

# Sample corpus
sentences = [
    "machine learning is fun",
    "deep learning is part of machine learning",
    "natural language processing is a field of ai",
    "word embeddings are learned representations",
    "tensorflow makes it easy to build models"
]

# Tokenize Corpus
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(sentences)
word2idx = tokenizer.word_index
idx2word = {v: k for k, v in word2idx.items()}
vocab_size = len(word2idx) + 1  # Fixed typo: voacb_size -> vocab_size

# Generate Skip-gram pairs
window_size = 2
sequences = tokenizer.texts_to_sequences(sentences)
pairs = []
for seq in sequences:
    for i, target_word in enumerate(seq):
        context_window = seq[max(i - window_size, 0): i] + seq[i + 1: i + window_size + 1]
        for context_word in context_window:
            pairs.append((target_word, context_word))

# Convert to numpy arrays
targets, contexts = zip(*pairs)
targets = np.array(targets)
contexts = np.array(contexts)
 
# One-hot encode targets
context_labels = tf.keras.utils.to_categorical(contexts, num_classes=vocab_size)
 
# Define skip-gram model
embedding_dim = 64
input_word = tf.keras.Input(shape=(1,))

# Added explicit name='embedding' here to match your lookup strategy
embedding = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, name='embedding')(input_word)

x = tf.keras.layers.Reshape((embedding_dim,))(embedding)
output = tf.keras.layers.Dense(vocab_size, activation='softmax')(x)
 
model = tf.keras.Model(inputs=input_word, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy')
 
# Train the model
model.fit(targets, context_labels, epochs=100, verbose=0)
 
# Extract and display learned embeddings
embedding_weights = model.get_layer('embedding').get_weights()[0]
for word, idx in word2idx.items():
    vec = embedding_weights[idx][:5]  # Show first 5 dims
    print(f"{word}: {vec.round(3)}")

### 35.Sequence-to-Sequence Translator (English﷿﷿﷿French)

In [ ]:
# Build a basic Seq2Seq (Encoder–Decoder) model using LSTM layers in TensorFlow 2 to translate short English sentences into French.

import tensorflow as tf
import numpy as np

# Sample parallel corpus
english_sentences = ["hello", "how are you", "thank you", "good night"]
french_sentences = ["bonjour", "comment ça va", "merci", "bonne nuit"]

# Tokenize source (English)
src_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
src_tokenizer.fit_on_texts(english_sentences)
src_sequences = src_tokenizer.texts_to_sequences(english_sentences)
src_word_index = src_tokenizer.word_index
src_vocab_size = len(src_word_index) + 1

# Tokenize target (French) and add <start>, <end> tokens
french_sentences = [f"<start> {s} <end>" for s in french_sentences]
tgt_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tgt_tokenizer.fit_on_texts(french_sentences)
tgt_sequences = tgt_tokenizer.texts_to_sequences(french_sentences)
tgt_word_index = tgt_tokenizer.word_index
tgt_vocab_size = len(tgt_word_index) + 1

# Pad sequences
src_padded = tf.keras.preprocessing.sequence.pad_sequences(src_sequences, padding='post')
tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(tgt_sequences, padding='post')

# Split target into decoder input and output
decoder_input = tgt_padded[:, :-1]
decoder_target = tf.keras.utils.to_categorical(tgt_padded[:, 1:], num_classes=tgt_vocab_size)

# ================= ENCODER TRAINING LAYERS =================
embedding_dim = 64
encoder_inputs = tf.keras.Input(shape=(None,), name="encoder_inputs")
encoder_embedding = tf.keras.layers.Embedding(src_vocab_size, embedding_dim, name="encoder_embedding")
x = encoder_embedding(encoder_inputs)

encoder_lstm = tf.keras.layers.LSTM(64, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(x)
encoder_states = [state_h, state_c]

# ================= DECODER TRAINING LAYERS =================
decoder_inputs = tf.keras.Input(shape=(None,), name="decoder_inputs")
decoder_embedding = tf.keras.layers.Embedding(tgt_vocab_size, embedding_dim, name="decoder_embedding")
y = decoder_embedding(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(64, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(y, initial_state=encoder_states)

decoder_dense = tf.keras.layers.Dense(tgt_vocab_size, activation='softmax', name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# Train the end-to-end model
model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit([src_padded, decoder_input], decoder_target, epochs=300, verbose=0)


# ================= INFERENCE MODELS =================
# 1. Define the standalone encoder model to extract thoughts/states
inf_encoder_model = tf.keras.Model(encoder_inputs, encoder_states)

# 2. Define the standalone decoder model that accepts states step-by-step
decoder_state_input_h = tf.keras.Input(shape=(64,))
decoder_state_input_c = tf.keras.Input(shape=(64,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Re-use the trained layers from above
inf_decoder_inputs = tf.keras.Input(shape=(1,)) # one token at a time
y_inf = decoder_embedding(inf_decoder_inputs)
inf_decoder_outputs, inf_state_h, inf_state_c = decoder_lstm(y_inf, initial_state=decoder_states_inputs)
inf_decoder_states = [inf_state_h, inf_state_c]
inf_decoder_outputs = decoder_dense(inf_decoder_outputs)

inf_decoder_model = tf.keras.Model(
    [inf_decoder_inputs] + decoder_states_inputs,
    [inf_decoder_outputs] + inf_decoder_states
)

# ================= INFERENCE TRANSLATION FUNCTION =================
def translate(input_text):
    # Preprocess input phrase
    seq = src_tokenizer.texts_to_sequences([input_text])
    seq = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=src_padded.shape[1], padding='post')
    
    # Get initial context states from encoder
    states_value = inf_encoder_model.predict(seq, verbose=0)
    
    # Generate start token sequence (<start>)
    target_seq = np.array([[tgt_word_index['<start>']]])
    
    translated = []
    for _ in range(10):
        # Predict next token distribution and updated states
        output_tokens, h, c = inf_decoder_model.predict([target_seq] + states_value, verbose=0)
        
        # Sample the token with highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = tgt_tokenizer.index_word.get(sampled_token_index, '')
        
        if sampled_word == '<end>' or not sampled_word:
            break
            
        translated.append(sampled_word)
        
        # Update the target token and states for the next iteration
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]
        
    return ' '.join(translated)

# Test translation
print("Translate 'thank you':", translate("thank you"))


In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

# 1. Load the Dataset (Using WMT14 - universally supported for fr-en)
examples, metadata = tfds.load(
    'wmt14_translate/fr-en', 
    with_info=True, 
    as_supervised=True
)
train_examples = examples['train']

BATCH_SIZE = 64
BUFFER_SIZE = 20000

# WMT14 yields (french_tensor, english_tensor) as supervised pairs
def format_dataset(fr, en):
    en_text = en  # English Source
    fr_text = tf.strings.join(['<start> ', fr, ' <end>']) # French Target
    return en_text, fr_text

train_dataset = train_examples.map(format_dataset).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

# 2. Vectorization (Tokenization & Padding)
VOCAB_SIZE = 5000
MAX_LEN = 20

# English (Source) Vectorizer
src_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
src_vectorizer.adapt(train_dataset.map(lambda src, tgt: src).take(100)) # Quick adapt over 100 batches

# French (Target) Vectorizer
@tf.keras.utils.register_keras_serializable()
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, r"[^a-z<>?]", " ")

tgt_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN + 1,
    standardize=custom_standardization
)
tgt_vectorizer.adapt(train_dataset.map(lambda src, tgt: tgt).take(100))

# 3. Final Pipeline Processing for Training
def process_sequences(src_text, tgt_text):
    src_tokens = src_vectorizer(src_text)
    tgt_tokens = tgt_vectorizer(tgt_text)
    
    dec_input = tgt_tokens[:, :-1]
    dec_target = tgt_tokens[:, 1:]
    
    return (src_tokens, dec_input), dec_target

train_dataset = train_dataset.map(process_sequences).prefetch(tf.data.AUTOTUNE)

# 4. Define the Seq2Seq Model Architectures
embedding_dim = 128
latent_dim = 256

# ================= ENCODER LAYERS =================
encoder_inputs = tf.keras.Input(shape=(None,), name="encoder_inputs")
encoder_embedding = tf.keras.layers.Embedding(VOCAB_SIZE, embedding_dim, mask_zero=True, name="encoder_embedding")
x = encoder_embedding(encoder_inputs)

encoder_lstm = tf.keras.layers.LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(x)
encoder_states = [state_h, state_c]

# ================= DECODER LAYERS =================
decoder_inputs = tf.keras.Input(shape=(None,), name="decoder_inputs")
decoder_embedding = tf.keras.layers.Embedding(VOCAB_SIZE, embedding_dim, mask_zero=True, name="decoder_embedding")
y = decoder_embedding(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(y, initial_state=encoder_states)

decoder_dense = tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax', name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# Train End-to-End Model
model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Quick training run over a portion of the dataset
print("Training model...")
model.fit(train_dataset.take(50), epochs=15)


# ================= INFERENCE SETUP =================
inf_encoder_model = tf.keras.Model(encoder_inputs, encoder_states)

decoder_state_input_h = tf.keras.Input(shape=(latent_dim,))
decoder_state_input_c = tf.keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

inf_decoder_inputs = tf.keras.Input(shape=(1,)) 
y_inf = decoder_embedding(inf_decoder_inputs)
inf_decoder_outputs, inf_state_h, inf_state_c = decoder_lstm(y_inf, initial_state=decoder_states_inputs)
inf_decoder_states = [inf_state_h, inf_state_c]
inf_decoder_outputs = decoder_dense(inf_decoder_outputs)

inf_decoder_model = tf.keras.Model(
    [inf_decoder_inputs] + decoder_states_inputs,
    [inf_decoder_outputs] + inf_decoder_states
)

# Extract French vocabulary for generation decoding
tgt_vocab = tgt_vectorizer.get_vocabulary()

def translate(input_text):
    seq = src_vectorizer([input_text])
    states_value = inf_encoder_model.predict(seq, verbose=0)
    
    try:
        start_index = tgt_vocab.index('<start>')
        end_index = tgt_vocab.index('<end>')
    except ValueError:
        return "Error: `<start>` or `<end>` tokens missing from vocabulary map."
    
    target_seq = np.array([[start_index]])
    translated = []
    
    for _ in range(MAX_LEN):
        output_tokens, h, c = inf_decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        
        if sampled_token_index == end_index or sampled_token_index == 0: 
            break
            
        sampled_word = tgt_vocab[sampled_token_index]
        translated.append(sampled_word)
        
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]
        
    return ' '.join(translated)

# Test translation output (English -> French)
print("\n--- Translation Test ---")
print("English: 'hello'")
print("French Prediction:", translate("hello"))

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

...

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

: 

: 

: 

### 36. GRU-based Language Model

In [2]:
# Train a character-level GRU language model using TensorFlow 2 to generate text one character at a time, based on a short input sequence.

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load and prepare text (tiny Shakespeare excerpt sample)
text = "To be, or not to be, that is the question."
chars = sorted(set(text))
char2idx = {ch: i for i, ch in enumerate(chars)} # char2idx means character to index mapping and Char2idx changes each character to a unique integer index
idx2char = np.array(chars) # idx2char means index to character mapping and idx2char changes each integer index back to its corresponding character
vocab_size = len(chars)

# Convert entire text to imterger sequence
text_as_int = np.array([char2idx[c] for c in text])

# Create input-output pairs for training (seq -> next char)
seq_length = 10
examples_per_epoch = len(text_as_int) - seq_length
inputs = []
targets = []
for i in range(examples_per_epoch):
    inputs.append(text_as_int[i:i+seq_length])
    targets.append(text_as_int[i+1:i+seq_length+1])

X = np.array(inputs)
y = np.array(targets)

# Build GRU Language model
embedding_dim = 32
gru_units = 64

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=seq_length), # Character embedding layer
    tf.keras.layers.GRU(gru_units, return_sequences=True), # GRU layer
    tf.keras.layers.Dense(vocab_size, activation='softmax') # Output layer for next character prediction
])

# Compile the model and train
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(X, y, epochs=200, verbose=0)

# Generate text from a seed sequence
def generate_text(seed, length=100):
    input_eval = [char2idx[c] for c in seed.lower()] # convert seed to integer sequence
    input_eval = tf.expand_dims(input_eval, 0) # add batch dimension
    result = list(seed) # start result with seed characters
    
    for _ in range(length):
        predictions = model(input_eval) # get predictions for the next character
        predicted_id = tf.argmax(predictions[0, -1]).numpy() # get the index of the most likely next character
        result.append(idx2char[predicted_id]) # append the predicted character to the result
        
        input_eval = tf.expand_dims([predicted_id], 0) # update input for the next prediction
    
    return ''.join(result)

# # Test text generation
# seed_text = "To be, or "
# generated_text = generate_text(seed_text, length=100)
# print("Generated Text:\n", generated_text)
# Generate from seed
print(generate_text("To be, or "))


To be, or no t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t t


### 37. Named Entity Recognition with Bi-LSTM

In [5]:
# Build a Bi-LSTM model for Named Entity Recognition (NER) using TensorFlow 2 on a small manually defined dataset. The model predicts tags like PER (person), LOC (location), or O (other).

import tensorflow as tf
import numpy as np
 
# Sample token-level NER dataset
sentences = [["john", "lives", "in", "new", "york"],
             ["alice", "is", "from", "paris"],
             ["bob", "visited", "london", "last", "year"]]
 
labels = [["PER", "O", "O", "LOC", "LOC"],
          ["PER", "O", "O", "LOC"],
          ["PER", "O", "LOC", "O", "O"]]

# Build Vocabularies
word_tokenizer = tf.keras.preprocessing.text.Tokenizer(lower=True, oov_token='UNK')
# here word_tokenizer is a here word _ tokenizer is a tokenizer  that will be used to convert the sentences into sequences of integers. The 'Lower—True' argument means that alt words will be converted to lowercase before tokenization, and 'UNk' ' means that any word not found in the training data wilt be represented as 'UNK' (unknown).
word_tokenizer.fit_on_texts(sentences)
X = word_tokenizer.texts_to_sequences(sentences)
word_index = word_tokenizer.word_index
vocan_size = len(word_index) + 1

# here we are but Idin a ta tokenizer that will be used to convert the labels into sequences of integers. The 'Lower=False' argument means that the tags will not be converted to lowercase before tokenization, and 'O' means that any tag not found in the training data will be represented as 'O' (other)
tag_tokenizer = tf.keras.preprocessing.text.Tokenizer(lower=False)
tag_tokenizer.fit_on_texts(labels)
y = tag_tokenizer.texts_to_sequences(labels)
tag_index = tag_tokenizer.word_index
num_tags = len(tag_index) + 1

# Pad sequences
max_len = max(len(s) for s in X)
X = tf.keras.preprocessing.sequence.pad_sequences(X, maxlen=max_len, padding='post')
y = tf.keras.preprocessing.sequence.pad_sequences(y, maxlen=max_len, padding='post')

# Convert labels to categorical
y_cat = tf.keras.utils.to_categorical(y, num_classes=num_tags)

# Build Bi-LSTM model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=64,input_length=max_len),  # Word embeddings
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),        # Bi-LSTM
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(num_tags, activation='softmax')) # One output per token
])

# Compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train
model.fit(X, y_cat, epochs=50, verbose=0)

# Predict on a new sentence
test_sentence = ["alice", "visited", "new", "delhi"]
test_seq = word_tokenizer.texts_to_sequences([test_sentence])
test_seq = tf.keras.preprocessing.sequence.pad_sequences(test_seq, maxlen=max_len, padding='post')

pred = model.predict(test_seq)[0]
pred_tags = [list(tag_index.keys())[np.argmax(p) - 1] if np.argmax(p) > 0 else "PAD" for p in pred]

# Print prediction
for word, tag in zip(test_sentence, pred_tags):
    print(f"{word} → {tag}")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
alice → PER
visited → O
new → O
delhi → LOC
